In [1]:
!pip install "transformers[torch]"

In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration, DataCollatorForSeq2Seq

In [5]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
train_data.loc[0]["dialogue"]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [8]:
train_data.loc[0]["summary"]

'Amanda baked cookies and will bring Jerry some tomorrow.'

In [9]:
train_data.shape

(14732, 3)

In [10]:
val_data.shape

(818, 3)

In [11]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [12]:
train_data.shape

(4000, 3)

## Data pre-processing

In [4]:
import re

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = re.sub(r"\s+", " ", text)     # collapse spaces/newlines
    text = text.strip()
    return text

In [14]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [15]:
train_data.loc[0]["dialogue"]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet: Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## Tokenize

In [16]:
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [17]:
# raw data => tokenized inputs for fine-tuning

# 1. tokenize: one row at a time (for pandas apply)
def tokenize(data):
    dialogue = "summarize: " + clean_data(data["dialogue"])
    summary = clean_data(data["summary"])

    inputs = tokenizer(dialogue, max_length=512, truncation=True)
    targets = tokenizer(text_target=summary, max_length=128, truncation=True)

    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"],
    }

In [18]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [19]:
# 2. sanity check: must print <class 'int'> and 0
print(type(train_dataset[0]["input_ids"][0]))
print("nested:", sum(isinstance(e["input_ids"][0], list) for e in train_dataset))

<class 'int'>
nested: 0


In [20]:
train_dataset[0]

{'input_ids': [21603,
  10,
  28866,
  10,
  7102,
  55,
  3,
  23,
  764,
  640,
  48,
  8513,
  31,
  7,
  1108,
  11,
  3,
  23,
  816,
  24,
  25,
  429,
  253,
  34,
  1477,
  28866,
  10,
  19542,
  10,
  2018,
  55,
  3,
  10,
  61,
  1333,
  6,
  68,
  27,
  31,
  162,
  641,
  608,
  34,
  5,
  3,
  10,
  61,
  19542,
  10,
  299,
  2049,
  21,
  1631,
  81,
  140,
  3,
  10,
  61,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [28866, 1622, 19542, 8513, 31, 7, 1108, 5, 1]}

In [21]:
len(train_dataset[0]["input_ids"])

59

## Working with our Model

In [22]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [24]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=5,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    warmup_steps=100
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [26]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.448928,1.405073
2,1.349666,1.403164
3,1.273250,1.415381
4,1.219669,1.417239


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,1.448928,1.405073
2,1.349666,1.403164
3,1.273250,1.415381
4,1.219669,1.417239
5,1.187288,1.420456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=2500, training_loss=1.295759912109375, metrics={'train_runtime': 2314.2735, 'train_samples_per_second': 8.642, 'train_steps_per_second': 1.08, 'total_flos': 8754815339053056.0, 'train_loss': 1.295759912109375, 'epoch': 5.0})

In [27]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [3]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
model.to(device)
model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

## Test the core logic for summarization

In [8]:
def summarize_dialogue(dialogue):
  dialogue = "summarize: " + clean_data(dialogue) # clean

  # tokenize
  inputs = tokenizer(
      dialogue,
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  # generate the summary => token ids
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=128,
      num_beams=4,
      early_stopping=True
  )

  # token ids convert to summary => decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)
  return summary

In [13]:
test_dialogue = """
Aditi: You said you'd call me yesterday. I waited the whole evening.
Rohan: I'm sorry, I got stuck at work till 11.
Aditi: You could have at least texted.
Rohan: You're right, I should have. Can I make it up to you with dinner tonight?
Aditi: Fine, but you're paying.
Rohan: Deal.
"""
summary = summarize_dialogue(test_dialogue)
print(summary)

Aditi waited for Rohan to call her yesterday. Rohan got stuck at work till 11. He will make it up to Aditi with dinner tonight.
